In [1]:
from planning.planner import (
    connect_poses_with_curvature,
    pose_to_node,
    node_to_pose,
    plan_from_path,
    a_star,
)
from planning.common import node_to_pose, pose_to_node, plan_to_destination_dt
from planning.timed_env import TimedEnv, generate_trajectory
from planning.collision_check import check_point_collision
import networkx as nx
import matplotlib.pyplot as plt
from aido_schemas import Context, FriendlyPose
from dt_protocols import PlanningSetup, Rectangle, Circle, PlacedPrimitive, Appearance
import math
import numpy as np

from aido_schemas import FriendlyPose
from dt_protocols import (
    PlanningSetup, 
    Rectangle, 
    Circle, 
    PlacedPrimitive, 
    Appearance, 
    Motion, 
    PlanStep
)
import math

motion1 = [
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
]

motion2 = [
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
    PlanStep(duration=3.0, velocity_x_m_s=-0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=-450.0),
]

motion3 = [
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
]

motion4 = [
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
    PlanStep(duration=3.0, velocity_x_m_s=0.13333333333333333, angular_velocity_deg_s=0.0),
    PlanStep(duration=0.2, velocity_x_m_s=0.0, angular_velocity_deg_s=450.0),
]

# Create the environment from the log data
environment = [
    # Static walls (brown rectangles)
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=0.0, ymin=0.0, xmax=0.1, ymax=4.0),
        appearance=Appearance(fillcolor="brown")
    ),
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=3.9, ymin=0.0, xmax=4.0, ymax=4.0),
        appearance=Appearance(fillcolor="brown")
    ),
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=0.0, ymin=3.9, xmax=4.0, ymax=4.0),
        appearance=Appearance(fillcolor="brown")
    ),
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=0.0, ymin=0.0, xmax=4.0, ymax=0.1),
        appearance=Appearance(fillcolor="brown")
    ),
    
    # Moving obstacles (yellow circles with motion)
    # Object #4
    PlacedPrimitive(
        pose=FriendlyPose(x=0.2, y=0.6, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #5
    PlacedPrimitive(
        pose=FriendlyPose(x=0.2, y=1.4, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #6
    PlacedPrimitive(
        pose=FriendlyPose(x=0.2, y=2.2, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #7
    PlacedPrimitive(
        pose=FriendlyPose(x=0.2, y=2.6, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #8
    PlacedPrimitive(
        pose=FriendlyPose(x=0.2, y=3.4, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #9
    PlacedPrimitive(
        pose=FriendlyPose(x=1.0, y=0.2, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #10
    PlacedPrimitive(
        pose=FriendlyPose(x=1.0, y=1.0, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #11
    PlacedPrimitive(
        pose=FriendlyPose(x=1.0, y=1.8, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #12
    PlacedPrimitive(
        pose=FriendlyPose(x=1.0, y=3.0, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #13
    PlacedPrimitive(
        pose=FriendlyPose(x=1.0, y=3.4, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #14
    PlacedPrimitive(
        pose=FriendlyPose(x=1.8, y=0.2, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #15
    PlacedPrimitive(
        pose=FriendlyPose(x=1.8, y=1.0, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion2,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #16
    PlacedPrimitive(
        pose=FriendlyPose(x=2.2, y=1.8, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion3,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #18
    PlacedPrimitive(
        pose=FriendlyPose(x=1.8, y=3.8, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #19
    PlacedPrimitive(
        pose=FriendlyPose(x=2.6, y=0.2, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #20
    PlacedPrimitive(
        pose=FriendlyPose(x=2.6, y=1.0, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #21
    PlacedPrimitive(
        pose=FriendlyPose(x=2.6, y=1.8, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #22
    PlacedPrimitive(
        pose=FriendlyPose(x=2.6, y=3.0, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #23
    PlacedPrimitive(
        pose=FriendlyPose(x=2.6, y=3.4, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion1,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #24
    PlacedPrimitive(
        pose=FriendlyPose(x=3.4, y=0.2, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion2,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #25
    PlacedPrimitive(
        pose=FriendlyPose(x=3.4, y=1.0, theta_deg=-90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion2,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #26
    PlacedPrimitive(
        pose=FriendlyPose(x=3.4, y=2.2, theta_deg=-180.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion2,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #27
    PlacedPrimitive(
        pose=FriendlyPose(x=3.8, y=2.6, theta_deg=90.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion4,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
    
    # Object #28
    PlacedPrimitive(
        pose=FriendlyPose(x=3.8, y=3.4, theta_deg=0.0),
        primitive=Circle(radius=0.05),
        motion=Motion(
            steps=motion3,
            periodic=True
        ),
        appearance=Appearance(fillcolor="yellow")
    ),
]

# Robot body (from the log)
body = [
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=-0.13, ymin=-0.045, xmax=0.07, ymax=0.045),
        appearance=Appearance(fillcolor="blue", rel_zorder=1)
    ),
    PlacedPrimitive(
        pose=FriendlyPose(x=0.0, y=0.0, theta_deg=0.0),
        primitive=Rectangle(xmin=-0.03, ymin=-0.065, xmax=0.03, ymax=0.065),
        appearance=Appearance(fillcolor="black", rel_zorder=-1)
    ),
]

# Create the PlanningSetup
ps = PlanningSetup(
    environment=environment,
    body=body,
    bounds=Rectangle(xmin=0.0, ymin=0.0, xmax=4.0, ymax=4.0),
    max_linear_velocity_m_s=0.4,
    min_linear_velocity_m_s=-0.3,
    max_angular_velocity_deg_s=30.0,
    max_curvature=float("inf"),
    tolerance_xy_m=0.05,
    tolerance_theta_deg=20.0,
)

t_env = TimedEnv(ps.environment, 0.1)

DEBUG:commons:version: 6.2.4 *
DEBUG:typing:version: 6.2.3
DEBUG:dt_protocols:dt-protocols version 6.2.35 path /usr/local/lib/python3.8/dist-packages
DEBUG:nodes:version 6.2.17 path /usr/local/lib/python3.8/dist-packages pyparsing 2.4.6
DEBUG:duckietown_world:duckietown-world version 6.4.3 path /usr/local/lib/python3.8/dist-packages
DEBUG:geometry:PyGeometry-z6 version 2.1.4 path /usr/local/lib/python3.8/dist-packages
DEBUG:aido_schemas:aido-protocols version 6.1.1 path /usr/local/lib/python3.8/dist-packages
DEBUG:duckietown_challenges:duckietown_challenges version 6.5.2 path /usr/local/lib/python3.8/dist-packages
DEBUG:duckietown_build_utils:duckietown_build_utils version 6.2.78 path /usr/local/lib/python3.8/dist-packages
DEBUG:duckietown_docker_utils:duckietown_docker_utils version 6.1.1 path /usr/local/lib/python3.8/dist-packages
DEBUG:ipce:version 6.1.2 path /usr/local/lib/python3.8/dist-packages


In [2]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_plan(start_pose, plan):
    """
    Simulate the trajectory given a start pose (FriendlyPose) and a plan (list of PlanStep).
    Returns arrays of x, y, theta.
    """
    x, y, theta_deg = start_pose.x, start_pose.y, start_pose.theta_deg
    xs, ys, thetas = [x], [y], [theta_deg]
    for step in plan:
        duration = step.duration
        v = step.velocity_x_m_s
        w_deg = step.angular_velocity_deg_s
        w = np.deg2rad(w_deg)
        theta = np.deg2rad(theta_deg)
        dt = 0.001
        n = max(1, int(duration / dt))
        for _ in range(n):
            if abs(w) < 1e-6:
                x += v * np.cos(theta) * dt
                y += v * np.sin(theta) * dt
            else:
                x += v * np.cos(theta) * dt
                y += v * np.sin(theta) * dt
                theta += w * dt
            xs.append(x)
            ys.append(y)
            thetas.append(np.rad2deg(theta))
        theta_deg = np.rad2deg(theta)
    return np.array(xs), np.array(ys), np.array(thetas)

def plot_trajectory_from_path(plan, path, planning_setup=None):
    """
    Plots the continuous trajectory defined by the plans on the edges of the path.
    The line is blue if the vehicle moves forward, green if it moves backward.
    If planning_setup is provided, also plots the environment.
    """

    def draw_primitive(ax, placed_primitive):
        pose = placed_primitive.pose
        primitive = placed_primitive.primitive
        appearance = getattr(placed_primitive, "appearance", None)
        color = getattr(appearance, "fillcolor", "gray") if appearance else "gray"
        if isinstance(primitive, Rectangle):
            # Rectangle is defined in local frame, so we need to transform corners
            corners = np.array([
                [primitive.xmin, primitive.ymin],
                [primitive.xmax, primitive.ymin],
                [primitive.xmax, primitive.ymax],
                [primitive.xmin, primitive.ymax],
                [primitive.xmin, primitive.ymin],
            ])
            # Rotation
            theta = np.deg2rad(pose.theta_deg)
            R = np.array([[np.cos(theta), -np.sin(theta)],
                          [np.sin(theta),  np.cos(theta)]])
            rotated = (R @ corners.T).T
            # Translation
            translated = rotated + np.array([pose.x, pose.y])
            ax.plot(translated[:,0], translated[:,1], color=color, linewidth=0)
            ax.fill(translated[:,0], translated[:,1], color=color, alpha=0.8)
        elif isinstance(primitive, Circle):
            circle = plt.Circle((pose.x, pose.y), primitive.radius, color=color, alpha=0.8, linewidth=0)
            ax.add_patch(circle)

    start_node = path[0]
    start_pose = node_to_pose(start_node)
    x, y, theta_deg = start_pose.x, start_pose.y, start_pose.theta_deg

    fig, ax = plt.subplots(figsize=(6,6))
    xs_all, ys_all = [x], [y]  # For start/goal markers

    # Draw environment if provided
    if planning_setup is not None:
        for placed_primitive in getattr(planning_setup, "environment", []):
            draw_primitive(ax, placed_primitive)

    for step in plan:
        # Simulate this step only
        temp_pose = type(start_pose)(x, y, theta_deg)
        xs, ys, thetas = simulate_plan(temp_pose, [step])
        # Only plot the segment (not the repeated start)
        color = 'blue' if step.velocity_x_m_s > 0 else 'green'
        ax.plot(xs, ys, color=color)
        # Update for next step
        x, y, theta_deg = xs[-1], ys[-1], thetas[-1]
        xs_all.extend(xs[1:])  # skip the first point to avoid duplicates
        ys_all.extend(ys[1:])

    ax.scatter([xs_all[0], xs_all[-1]], [ys_all[0], ys_all[-1]], c='red', label="Start/Goal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title("Planned Trajectory (from edge plans)")
    ax.axis('equal')
    ax.grid(True)
    ax.legend()
    plt.show()

In [3]:
# Add this new cell after your existing code

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, VBox, HBox, Label
import matplotlib.patches as patches

trajectory = []

def create_interactive_trajectory_viewer(plan, path, t_env, planning_setup=None):
    """
    Creates an interactive widget that allows users to explore the trajectory over time.
    
    Args:
        plan: List of PlanStep objects (motion commands)
        path: List of FriendlyPose objects (waypoints)
        t_env: TimedEnv object for getting environment at different times
        planning_setup: PlanningSetup object for environment visualization
    """
    
    def draw_primitive(ax, placed_primitive):
        """Helper function to draw a primitive on the given axes"""
        pose = placed_primitive.pose
        primitive = placed_primitive.primitive
        appearance = getattr(placed_primitive, "appearance", None)
        color = getattr(appearance, "fillcolor", "gray") if appearance else "gray"
        
        if isinstance(primitive, Rectangle):
            # Rectangle is defined in local frame, so we need to transform corners
            corners = np.array([
                [primitive.xmin, primitive.ymin],
                [primitive.xmax, primitive.ymin],
                [primitive.xmax, primitive.ymax],
                [primitive.xmin, primitive.ymax],
                [primitive.xmin, primitive.ymin],
            ])
            # Rotation
            theta = np.deg2rad(pose.theta_deg)
            R = np.array([[np.cos(theta), -np.sin(theta)],
                          [np.sin(theta),  np.cos(theta)]])
            rotated = (R @ corners.T).T
            # Translation
            translated = rotated + np.array([pose.x, pose.y])
            ax.plot(translated[:,0], translated[:,1], color=color, linewidth=0)
            ax.fill(translated[:,0], translated[:,1], color=color, alpha=0.8)
        elif isinstance(primitive, Circle):
            circle = plt.Circle((pose.x, pose.y), primitive.radius, color=color, alpha=0.8, linewidth=0)
            ax.add_patch(circle)
    
    def draw_robot_body(ax, pose, body_primitives):
        """Helper function to draw the robot body at a given pose"""
        for body_prim in body_primitives:
            # Create a copy of the body primitive with the current pose
            robot_prim = type(body_prim)(
                pose=pose,
                primitive=body_prim.primitive,
                appearance=body_prim.appearance
            )
            draw_primitive(ax, robot_prim)
    
    def get_vehicle_position_at_time(trajectory, t):
        """Get the vehicle position at time t by simulating the plan up to that time"""
        for dt, placed_primitive in trajectory:
            if dt >= t:
                pose = placed_primitive.pose
                break
        return FriendlyPose(x=pose.x, y=pose.y, theta_deg=pose.theta_deg)
    
    def plot_at_time(t):
        """Plot the environment and vehicle position at time t"""
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # Get environment at time t
        env_at_t = t_env.get_env(t)

        global trajectory
        if len(trajectory) == 0:
            p_prim = PlacedPrimitive(node_to_pose(path[0]), Circle(radius=1), Motion(steps=plan, periodic=False))
            trajectory = generate_trajectory(p_prim, t_env.dt)
        
        # Draw environment
        for placed_primitive in env_at_t:
            draw_primitive(ax, placed_primitive)
        
        # Get vehicle position at time t
        vehicle_pose = get_vehicle_position_at_time(trajectory, t)
        
        if vehicle_pose and planning_setup:
            # Draw robot body at current position
            draw_robot_body(ax, vehicle_pose, planning_setup.body)
        
        # Draw the full trajectory
        xs = list(map(lambda x: x[1].pose.x, trajectory))
        ys = list(map(lambda x: x[1].pose.y, trajectory))
        ax.plot(xs, ys, 'b-', alpha=0.3, linewidth=2, label='Planned trajectory')
        
        # Highlight current position
        if vehicle_pose:
            ax.scatter([vehicle_pose.x], [vehicle_pose.y], c='red', s=100, zorder=10, label='Current position')
        
        # Set plot properties
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        ax.set_title(f"Environment and Vehicle at t = {t:.2f}s")
        ax.axis('equal')
        ax.grid(True, alpha=0.3)
        ax.legend()
        
        # Set bounds based on planning setup if available
        if planning_setup:
            bounds = planning_setup.bounds
            ax.set_xlim(bounds.xmin - 0.1, bounds.xmax + 0.1)
            ax.set_ylim(bounds.ymin - 0.1, bounds.ymax + 0.1)
        
        plt.tight_layout()
        plt.show()
    
    # Calculate total plan duration
    total_duration = sum(step.duration for step in plan) if plan else 0
    
    # Create the interactive widget
    if total_duration > 0:
        time_slider = FloatSlider(
            value=0.0,
            min=0,
            max=total_duration,
            step=0.1,
            description='Time (s):',
            continuous_update=True
        )
        
        # Create the interactive plot
        interact(plot_at_time, t=time_slider)
    else:
        print("No plan available or plan has zero duration")

# Example usage (add this in a new cell):
# First, you'll need to run a_star to get a plan and path
# For example:
# start_pose = FriendlyPose(x=0.5, y=0.5, theta_deg=0)
# goal_pose = FriendlyPose(x=3.5, y=3.5, theta_deg=0)
# plan, path = a_star(ps, start_pose, goal_pose, t_env)

# Then create the interactive viewer:
# create_interactive_trajectory_viewer(plan, path, t_env, ps)

In [4]:
# Query 1
# start = FriendlyPose(0.6546626071393362, 0.578651154263448, 172.0221001932685)
# goal = FriendlyPose(3.4191263975032182, 3.540751120118101, 280.4175953067374)

# Query 2
start = FriendlyPose(3.0834961094151905, 0.3878856225788092, 200.37186763019042)
goal = FriendlyPose(0.584183154884033, 3.536243533232465, 186.80009078166805)

plan, path = a_star(ps, start, goal, t_env)
print(plan)

if plan is not None:
    create_interactive_trajectory_viewer(plan, path, t_env, ps)

/code/planning/packages/planning/planner.py:106: RuntimeWarning: divide by zero encountered in scalar divide
  if abs(theta / duration) > ps.max_angular_velocity_deg_s:


Suspicious start
A star finished, 124 nodes visited
[PlanStep(duration=3.0, velocity_x_m_s=0.13089969389957476, angular_velocity_deg_s=-30.0), PlanStep(duration=9.392729898879303, velocity_x_m_s=0.4, angular_velocity_deg_s=3.407881450573883), PlanStep(duration=1.4996896669119508, velocity_x_m_s=0.0, angular_velocity_deg_s=30.0)]


interactive(children=(FloatSlider(value=0.0, description='Time (s):', max=13.892419565791254), Output()), _dom…

In [5]:
print(path)

# Find the position of the vehicle at 7.7s
for dt, p_prim in trajectory:
    if dt >= 7.7:
        print(p_prim)
        break

# Get environment at 7.7s
env = t_env.get_env(7.7)

# Check collision
ps.environment = env
print(check_point_collision(ps, [pose_to_node(p_prim.pose)]))

# Check location at 7.7s based on path and plan

[(308, 39, 200), (276, 54, 110), (58, 354, 142.00930999264148), (58, 354, 187)]
PlacedPrimitive
│ pose: FriendlyPose(x=1.86, y=2.23, theta_deg=126)
│ primitive: Circle(radius=1)


TypeError: check_point_collision() missing 2 required positional arguments: 't' and 'points'

In [ ]:
plan[1].duration = 7.7 - plan[0].duration
destinations = plan_to_destination_dt(plan[1], path[1], 0.1, plan[0].duration)
print(destinations)
print(check_point_collision(ps, destinations))

[(306, 42, 128), (303, 45, 128), (301, 48, 128), (298, 51, 128), (296, 54, 128), (293, 57, 128), (291, 60, 128), (288, 64, 128), (286, 67, 128), (283, 70, 128), (281, 73, 128), (279, 76, 128), (276, 79, 128), (274, 82, 128), (271, 86, 128), (269, 89, 128), (266, 92, 128), (264, 95, 128), (261, 98, 128), (259, 101, 128), (256, 104, 128), (254, 107, 128), (251, 111, 128), (249, 114, 128), (246, 117, 128), (244, 120, 128), (241, 123, 128), (239, 126, 128), (236, 129, 128), (234, 133, 128), (231, 136, 128), (229, 139, 128), (226, 142, 128), (224, 145, 128), (221, 148, 128), (219, 151, 128), (216, 154, 128), (214, 158, 128), (211, 161, 128), (209, 164, 128), (206, 167, 128), (204, 170, 128), (201, 173, 128), (199, 176, 128), (196, 180, 128), (194, 183, 128), (191, 186, 128), (189, 189, 128), (187, 192, 128), (184, 195, 128), (182, 198, 128), (179, 201, 128), (177, 205, 128), (176, 206, 128)]
True
